In [1]:
import sys
import pandas as pd
import numpy as np
import sidrapy
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))
from src.soja_clima.soja import agregar_para_microrregiao

PROCESSED_DIR = Path("../../data/processed")

In [2]:
import time

print("Baixando dados do SIDRA via API (Tabela 1612) em lotes por ano...")

# Lista de anos (2000 a 2025) - ajuste se necessário
anos = [str(ano) for ano in range(2000, 2026)]

lista_dfs = []

for ano in anos:
    print(f"  -> Baixando ano {ano}...")
    try:
        df_ano = sidrapy.get_table(
            table_code="1612",
            territorial_level="6", # Municípios
            ibge_territorial_code="all",
            variable="all",
            classifications={"81": "2713"}, # Soja (em grão)
            period=ano
        )
        # Remove a linha de cabeçalho que o sidrapy retorna
        df_ano = df_ano.iloc[1:].reset_index(drop=True)
        lista_dfs.append(df_ano)
        time.sleep(0.5) # Pausa para não sobrecarregar a API
    except Exception as e:
        print(f"     ⚠️ Erro no ano {ano}: {e}")
        continue

# Concatena todos os anos em um único DataFrame
df_api = pd.concat(lista_dfs, ignore_index=True)

# Renomeia as colunas para o padrão do projeto
df_api = df_api.rename(columns={
    'D1C': 'codigo_muni',
    'D2C': 'ano',
    'D3N': 'variavel', # <--- CORRIGIDO AQUI (era D4N)
    'V': 'valor'
})

# Limpa os dados
df_api['valor'] = pd.to_numeric(
    df_api['valor'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False).replace(['-', '...', 'X'], np.nan),
    errors='coerce'
)
df_api = df_api.dropna(subset=['codigo_muni', 'ano', 'valor'])
df_api['ano'] = df_api['ano'].astype(int)
df_api['codigo_muni'] = df_api['codigo_muni'].astype(int).astype(str)

# Padroniza o nome das variáveis
df_api['variavel'] = (
    df_api['variavel']
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('ã', 'a')
    .str.replace('ç', 'c')
    .str.replace('é', 'e')
    .str.replace('ó', 'o')
)

print(f"\n✅ Download concluído! Total de linhas: {len(df_api)}")
print(f"Variáveis encontradas: {df_api['variavel'].unique()}")

Baixando dados do SIDRA via API (Tabela 1612) em lotes por ano...
  -> Baixando ano 2000...
  -> Baixando ano 2001...
  -> Baixando ano 2002...
  -> Baixando ano 2003...
  -> Baixando ano 2004...
  -> Baixando ano 2005...
  -> Baixando ano 2006...
  -> Baixando ano 2007...
  -> Baixando ano 2008...
  -> Baixando ano 2009...
  -> Baixando ano 2010...
  -> Baixando ano 2011...
  -> Baixando ano 2012...
  -> Baixando ano 2013...
  -> Baixando ano 2014...
  -> Baixando ano 2015...
  -> Baixando ano 2016...
  -> Baixando ano 2017...
  -> Baixando ano 2018...
  -> Baixando ano 2019...
  -> Baixando ano 2020...
  -> Baixando ano 2021...
  -> Baixando ano 2022...
  -> Baixando ano 2023...
  -> Baixando ano 2024...
  -> Baixando ano 2025...

✅ Download concluído! Total de linhas: 427520
Variáveis encontradas: ['área_plantada' 'área_plantada_-_percentual_do_total_geral'
 'área_colhida' 'área_colhida_-_percentual_do_total_geral'
 'quantidade_produzida' 'rendimento_medio_da_producao' 'valor_da_pro

In [3]:
print("Agregando dados por microrregião...")
df_soja = agregar_para_microrregiao(df_api)

df_soja.to_parquet(PROCESSED_DIR / "soja_microrregiao.parquet", index=False)
print("Dados de soja salvos com sucesso em soja_microrregiao.parquet!")
display(df_soja.head())

Agregando dados por microrregião...


municipalities_2020_simplified.parquet: 100%|██████████| 20.2M/20.2M [00:01<00:00, 18.6MB/s]
microregions_2020_simplified.parquet: 100%|██████████| 7.91M/7.91M [00:00<00:00, 15.9MB/s]


Dados de soja salvos com sucesso em soja_microrregiao.parquet!


,CD_MICRO,NM_MICRO,ano,quantidade_produzida,valor_da_producao,área_colhida,área_plantada,rendimento_recalculado
0,11001,Porto Velho,2002,90.0,54.0,50.0,50.0,1800.0
1,11001,Porto Velho,2005,765.0,333.0,255.0,255.0,3000.0
2,11001,Porto Velho,2006,896.0,333.0,320.0,320.0,2800.0
3,11001,Porto Velho,2007,810.0,316.0,300.0,320.0,2700.0
4,11001,Porto Velho,2008,552.0,315.0,200.0,200.0,2760.0
